# Fase 3 — Integración multimodal (WSI + RNA-seq) mediante MOFA+

**Tesis:** Clasificación de subtipos de cáncer de mama (Luminal A, Luminal B, Basal)  
**Cohorte:** TCGA-BRCA pareada (n=552)  
**Validación externa:** INC Bogotá (Luminal B)  

## Pipeline
1. Carga y preprocesamiento de ambas modalidades
2. Matrices de similitud paciente-paciente + Test de Mantel
3. Entrenamiento MOFA+ (factores latentes compartidos y específicos)
4. Extracción de outputs MOFA (Z, W, R²)
5. Asociación factores ↔ subtipo PAM50 (ANOVA + FDR)
6. GSEA sobre pesos RNA (interpretación biológica — anexo)
7. **Clasificación supervisada con CV repetida** (KNN/RF/LR/SVM/XGBoost)
8. Visualizaciones
9. Validación externa INC
10. Comparación integral unimodal vs multimodal

In [2]:
# ============================================================
# CELDA 0: Instalación de dependencias
# ============================================================
%pip install statsmodels mofapy2 h5py gseapy umap-learn xgboost --quiet

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip list --format=freeze > requirements_mofa.txt
!grep -iE "^(numpy|scipy|scikit-learn|xgboost|mofapy2|gseapy|torch|timm)==" requirements_mofa.txt

Note: you may need to restart the kernel to use updated packages.
gseapy==1.2.1
mofapy2==0.7.4
numpy==1.26.4
scikit-learn==1.7.2
scipy==1.15.3
timm==0.9.16
torch==2.5.1+cu121
xgboost==3.1.3


In [4]:
"""
============================================================
CELDA 1: Imports y configuración global
============================================================
▸ Parametrizado para aceptar cualquier extractor WSI ganador
  de Fase 2 (KimiaNet-Mosaico, KimiaNet-Parches, UNI-Parches,
  KimiaNet-KMedoids, UNI2h-KMedoids).
▸ Cambiar WSI_EXTRACTOR_NAME y WSI_PATH según el ganador.
"""
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # backend no interactivo
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

import sklearn
from sklearn.preprocessing import normalize, LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    GridSearchCV, RandomizedSearchCV,
    RepeatedStratifiedKFold, StratifiedKFold, cross_validate
)
from sklearn.metrics import (
    f1_score, accuracy_score, adjusted_rand_score,
    confusion_matrix, classification_report
)
from scipy import stats
from scipy.spatial.distance import pdist, squareform, cosine
from statsmodels.stats.multitest import multipletests
from scipy.stats import spearmanr, wilcoxon

from mofapy2.run.entry_point import entry_point
import h5py

import xgboost
from xgboost import XGBClassifier

try:
    import gseapy as gp
    HAS_GSEAPY = True
except ImportError:
    HAS_GSEAPY = False
    print("AVISO: gseapy no disponible; GSEA se omitirá")

try:
    import umap.umap_ as umap_lib
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False

# ======================== CONFIGURACIÓN GLOBAL ========================
# ▼▼▼ CAMBIAR SEGÚN EL GANADOR DE FASE 2 ▼▼▼
WSI_EXTRACTOR_NAME = 'UNI_Parches'   # Nombre descriptivo del extractor WSI ganador
WSI_FEAT_DIM       = 1024                  # Dimensionalidad del vector de features WSI
WSI_FEAT_PREFIX    = 'uni'                 # Prefijo para nombres de features WSI

# Paths
EXPR_PATH  = '/home/dlaguna/Consensus_DEG_signature.tsv'
WSI_PATH   = '/home/dlaguna/TCGA_work/meanpool_features/UNI_patches_meanpool_552.csv'
SPLIT_PATH = '/home/dlaguna/split_transcriptomica_604.csv'
# W_WSI_PATH se usa si ya se tiene la matriz de similitud precalculada
# W_WSI_PATH = '/home/dlaguna/Fusion/imagenes/W_WSI_JS_552x552.npy'

OUT_DIR = Path(f'/home/dlaguna/Fusion/new/{WSI_EXTRACTOR_NAME}')
OUT_DIR.mkdir(parents=True, exist_ok=True)
MOFA_MODEL_PATH = OUT_DIR / 'mofa_model.hdf5'

# Paths INC (validación externa)
INC_RNA_PATH = '/home/dlaguna/filtered_normalized_data_INC_23.csv'
INC_WSI_PATH = '/home/dlaguna/TCGA_work/UNI/Validacion_UNI_Parches/meanpool_features/UNI_patches_meanpool_INC23.csv'  # ← UNI

# Reproducibilidad
SEED = 42
np.random.seed(SEED)

# CV config (mismo diseño que Fases 1 y 2)
N_SPLITS  = 5
N_REPEATS = 3           # repeats solo para el reporte final
N_ITER_RANDOM = 80      # iteraciones para RandomizedSearchCV

print(f"Extractor WSI: {WSI_EXTRACTOR_NAME}")
print(f"Output dir:    {OUT_DIR}")
print(f"CV tuning:     StratifiedKFold({N_SPLITS})")
print(f"CV reporte:    RepeatedStratifiedKFold({N_SPLITS}×{N_REPEATS})")

I0000 00:00:1790224393.493767 1539728 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Extractor WSI: UNI_Parches
Output dir:    /home/dlaguna/Fusion/new/UNI_Parches
CV tuning:     StratifiedKFold(5)
CV reporte:    RepeatedStratifiedKFold(5×3)


In [5]:
"""
============================================================
PASO 1: Carga y preprocesamiento
============================================================
- RNA: log2(FPKM+2) ya hecho → centrar por gen
- WSI: L2-normalizar → centrar por feature
- Verosimilitud Gaussiana para ambas vistas (MOFA)
"""
print("="*70); print("PASO 1: Carga y preprocesamiento"); print("="*70)

# ---- Metadatos y split ----
split = pd.read_csv(SPLIT_PATH)
patient_order = split['patient_id'].tolist()
subtypes_str  = split.set_index('patient_id').loc[patient_order, 'Subtype'].values
split_role    = split.set_index('patient_id').loc[patient_order, 'split'].values

le = LabelEncoder()
y = le.fit_transform(subtypes_str)   # Basal=0, LumA=1, LumB=2
train_mask = (split_role == 'train')
test_mask  = (split_role == 'test')

# ---- RNA: cargar y alinear ----
raw = pd.read_csv(EXPR_PATH, sep='\t')
patient_ids_row = raw.iloc[0, 1:].values
gene_names      = raw.iloc[1:, 0].values
expr_arr        = raw.iloc[1:, 1:].astype(float).values
X_rna_all = pd.DataFrame(expr_arr.T, index=patient_ids_row, columns=gene_names)
X_RNA_raw = X_rna_all.loc[patient_order].values

# RNA: centrar por gen (MOFA recomienda esto manualmente)
X_RNA_proc = X_RNA_raw - X_RNA_raw.mean(axis=0, keepdims=True)
n_genes = X_RNA_proc.shape[1]

print(f"  RNA preprocessed: {X_RNA_proc.shape}")
print(f"    mean post-center: {X_RNA_proc.mean():.6f} (debe ser ~0)")
print(f"    std: {X_RNA_proc.std():.4f}")

# ---- WSI: cargar, alinear, L2-normalize, centrar ----
df_wsi = pd.read_csv(WSI_PATH).set_index('ID')
feat_cols = [c for c in df_wsi.columns if c != 'ID']
X_WSI_raw = df_wsi.loc[patient_order, feat_cols].astype(float).values

X_WSI_l2 = normalize(X_WSI_raw, norm='l2', axis=1)
X_WSI_proc = X_WSI_l2 - X_WSI_l2.mean(axis=0, keepdims=True)

print(f"  WSI preprocessed: {X_WSI_proc.shape}")
print(f"    mean post-center: {X_WSI_proc.mean():.6f} (debe ser ~0)")
print(f"    std: {X_WSI_proc.std():.4f}")

print(f"\n  Train={train_mask.sum()}, Test={test_mask.sum()}")
print(f"  Subtipos: {dict(zip(le.classes_, np.bincount(y)))}")
print(f"  Train dist: {dict(zip(le.classes_, np.bincount(y[train_mask])))}")
print(f"  Test  dist: {dict(zip(le.classes_, np.bincount(y[test_mask])))}")

PASO 1: Carga y preprocesamiento
  RNA preprocessed: (552, 604)
    mean post-center: -0.000000 (debe ser ~0)
    std: 1.6664
  WSI preprocessed: (552, 1024)
    mean post-center: -0.000000 (debe ser ~0)
    std: 0.0245

  Train=386, Test=166
  Subtipos: {'Basal': 166, 'LumA': 191, 'LumB': 195}
  Train dist: {'Basal': 116, 'LumA': 134, 'LumB': 136}
  Test  dist: {'Basal': 50, 'LumA': 57, 'LumB': 59}


In [6]:
"""
============================================================
PASO 2: Construcción de matrices de similitud paciente-paciente
============================================================
- RNA: correlación de Pearson sobre la firma Consensus DEG
- WSI: similitud coseno sobre features L2-normalizadas
- Diagnóstico de estructura por subtipo (gap + Mann-Whitney U)
- Heatmaps ordenados por subtipo
"""
from scipy.stats import mannwhitneyu

print("="*70); print("PASO 2: Matrices de similitud paciente-paciente"); print("="*70)

# ---- Matriz de similitud RNA (Pearson) ----
S_rna = np.corrcoef(X_RNA_proc)  # (552, 552)
print(f"  S_rna (Pearson): {S_rna.shape}, rango [{S_rna.min():.3f}, {S_rna.max():.3f}]")

# ---- Matriz de similitud WSI (coseno) ----
D_wsi = squareform(pdist(X_WSI_l2, metric='cosine'))
S_wsi = 1.0 - D_wsi
print(f"  S_wsi (coseno):  {S_wsi.shape}, rango [{S_wsi.min():.3f}, {S_wsi.max():.3f}]")

D_rna = 1.0 - S_rna  # se reutiliza en el test de Mantel (PASO 3)

# ---- Remapeo a [0,1] para heatmaps y compatibilidad con fusión ----
W_rna_01 = (S_rna + 1) / 2
W_wsi_01 = (S_wsi + 1) / 2

# ---- Gap analysis + Mann-Whitney U por matriz y subtipo ----
subtypes_unique = ['Basal', 'LumA', 'LumB']
gap_results = {}

for label, S_mat in [('RNA', S_rna), ('WSI', S_wsi)]:
    gap_results[label] = {}
    print(f"\n  {label}:")
    for s in subtypes_unique:
        mask_s = subtypes_str == s
        mask_o = ~mask_s

        intra = S_mat[np.ix_(mask_s, mask_s)][np.triu_indices(mask_s.sum(), k=1)]
        inter = S_mat[np.ix_(mask_s, mask_o)].ravel()

        gap = intra.mean() - inter.mean()
        u_stat, p_mwu = mannwhitneyu(intra, inter, alternative='two-sided')

        gap_results[label][s] = {
            'n_patients': int(mask_s.sum()),
            'intra_mean': float(intra.mean()), 'inter_mean': float(inter.mean()),
            'gap': float(gap), 'u_statistic': float(u_stat),
            'p_value': float(p_mwu), 'significant_005': bool(p_mwu < 0.05),
        }

        print(f"    {s:5s}: intra={intra.mean():.4f}  inter={inter.mean():.4f}  "
              f"gap={gap:+.4f}  p(MWU)={p_mwu:.2e}  "
              f"{'SIG' if p_mwu < 0.05 else 'n.s.'}")

    gap_global = np.mean([gap_results[label][s]['gap'] for s in subtypes_unique])
    gap_results[label]['global_gap'] = float(gap_global)
    print(f"    {'':5s} gap global: {gap_global:.4f}")

with open(OUT_DIR / 'gap_analysis_mwu.json', 'w') as f:
    json.dump(gap_results, f, indent=2)

# ---- Heatmaps ordenados por subtipo ----
order_idx = np.argsort(subtypes_str)
subtypes_sorted = subtypes_str[order_idx]
boundaries = np.where(subtypes_sorted[:-1] != subtypes_sorted[1:])[0] + 1

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (label, W_mat) in zip(axes, [('W_RNA (Pearson)', W_rna_01), ('W_WSI (coseno)', W_wsi_01)]):
    W_sorted = W_mat[np.ix_(order_idx, order_idx)]
    sns.heatmap(W_sorted, ax=ax, cmap='RdYlBu_r', vmin=0, vmax=1,
                cbar_kws={'label': 'Similitud'}, xticklabels=False, yticklabels=False)
    for b in boundaries:
        ax.axhline(b, color='black', lw=1)
        ax.axvline(b, color='black', lw=1)
    tick_pos, tick_labels, start = [], [], 0
    for b in list(boundaries) + [len(subtypes_sorted)]:
        tick_pos.append((start + b) / 2)
        tick_labels.append(subtypes_sorted[start])
        start = b
    ax.set_xticks(tick_pos); ax.set_xticklabels(tick_labels, rotation=0)
    ax.set_yticks(tick_pos); ax.set_yticklabels(tick_labels, rotation=0)
    gap_g = gap_results[label.split()[0].replace('W_', '')]['global_gap']
    ax.set_title(f'{label}\ngap global = {gap_g:.4f}')

plt.tight_layout()
plt.savefig(OUT_DIR / 'similarity_heatmaps_by_subtype.png', dpi=150, bbox_inches='tight')
plt.show()

# ---- Guardar matrices ----
np.save(OUT_DIR / 'S_rna_552x552.npy', S_rna)
np.save(OUT_DIR / 'S_wsi_552x552.npy', S_wsi)
print(f"\n  Archivos: S_rna_552x552.npy, S_wsi_552x552.npy, gap_analysis_mwu.json, similarity_heatmaps_by_subtype.png")

PASO 2: Matrices de similitud paciente-paciente
  S_rna (Pearson): (552, 552), rango [-0.641, 1.000]
  S_wsi (coseno):  (552, 552), rango [-0.068, 1.000]

  RNA:
    Basal: intra=0.5103  inter=-0.2836  gap=+0.7938  p(MWU)=0.00e+00  SIG
    LumA : intra=0.1986  inter=-0.0668  gap=+0.2654  p(MWU)=0.00e+00  SIG
    LumB : intra=0.1834  inter=-0.0667  gap=+0.2501  p(MWU)=0.00e+00  SIG
          gap global: 0.4364

  WSI:
    Basal: intra=0.4192  inter=0.3722  gap=+0.0470  p(MWU)=4.74e-285  SIG
    LumA : intra=0.3921  inter=0.3712  gap=+0.0209  p(MWU)=1.92e-72  SIG
    LumB : intra=0.4006  inter=0.3823  gap=+0.0182  p(MWU)=4.26e-56  SIG
          gap global: 0.0287

  Archivos: S_rna_552x552.npy, S_wsi_552x552.npy, gap_analysis_mwu.json, similarity_heatmaps_by_subtype.png


In [7]:
"""
============================================================
PASO 3: Test de Mantel (global y por subtipo)
============================================================
- Correlación entre matrices de distancia D_rna, D_wsi (ya
  calculadas en PASO 2) con significancia por permutación.
"""
print("="*70); print("PASO 3: Test de Mantel"); print("="*70)

def mantel_test(D_rna_sub, D_wsi_sub, n_perm=9999, seed=SEED):
    n_sub = D_rna_sub.shape[0]
    triu = np.triu_indices(n_sub, k=1)
    d_rna_v = D_rna_sub[triu]
    d_wsi_v = D_wsi_sub[triu]

    r_obs, _ = stats.pearsonr(d_rna_v, d_wsi_v)

    rng = np.random.default_rng(seed)
    r_perm = np.zeros(n_perm)
    for i in range(n_perm):
        perm = rng.permutation(n_sub)
        r_perm[i] = stats.pearsonr(D_rna_sub[np.ix_(perm, perm)][triu], d_wsi_v)[0]

    p_val = (np.sum(r_perm >= r_obs) + 1) / (n_perm + 1)
    return {
        'n_patients': int(n_sub), 'n_pairs': int(len(d_rna_v)),
        'r_observed': float(r_obs), 'p_value': float(p_val),
        'r_squared_pct': float(r_obs**2 * 100),
        'significant_005': bool(p_val < 0.05),
    }

# ---- Test de Mantel global ----
mantel_result = mantel_test(D_rna, D_wsi)
print(f"\n  Test de Mantel (global, n={mantel_result['n_patients']}, {mantel_result['n_pairs']} pares):")
print(f"    r_obs  = {mantel_result['r_observed']:.4f}  (r²={mantel_result['r_squared_pct']:.1f}%)")
print(f"    p-valor = {mantel_result['p_value']:.6f}  (9999 permutaciones)")
print(f"    {'SIGNIFICATIVO' if mantel_result['significant_005'] else 'NO significativo'} (α=0.05)")
print(f"    → {'Justifica fusión multimodal' if mantel_result['significant_005'] else 'Modalidades no acopladas'}")

# ---- Test de Mantel estratificado por subtipo ----
print(f"\n  Test de Mantel por subtipo:")
mantel_stratified = {}
for s in ['Basal', 'LumA', 'LumB']:
    mask_s = subtypes_str == s
    res = mantel_test(D_rna[np.ix_(mask_s, mask_s)], D_wsi[np.ix_(mask_s, mask_s)])
    mantel_stratified[s] = res
    print(f"    {s:5s}: n={res['n_patients']:3d}  r={res['r_observed']:.4f}  "
          f"(r²={res['r_squared_pct']:.1f}%)  p={res['p_value']:.6f}  "
          f"{'SIG' if res['significant_005'] else 'n.s.'}")

with open(OUT_DIR / 'mantel_test.json', 'w') as f:
    json.dump({'global': mantel_result, 'by_subtype': mantel_stratified}, f, indent=2)
print(f"\n  Archivo: mantel_test.json")

PASO 3: Test de Mantel

  Test de Mantel (global, n=552, 152076 pares):
    r_obs  = 0.1117  (r²=1.2%)
    p-valor = 0.000100  (9999 permutaciones)
    SIGNIFICATIVO (α=0.05)
    → Justifica fusión multimodal

  Test de Mantel por subtipo:
    Basal: n=166  r=0.1778  (r²=3.2%)  p=0.001600  SIG
    LumA : n=191  r=0.0682  (r²=0.5%)  p=0.028300  SIG
    LumB : n=195  r=-0.0131  (r²=0.0%)  p=0.651700  n.s.

  Archivo: mantel_test.json


In [8]:
# Reconstruir los vectores triangulares (global) — D_rna, D_wsi vienen de PASO 2
triu_idx = np.triu_indices(D_rna.shape[0], k=1)
d_rna_vec = D_rna[triu_idx]
d_wsi_vec = D_wsi[triu_idx]

fig, ax = plt.subplots(figsize=(6, 6))

# Subsample opcional si hay demasiados puntos (552*551/2 ≈ 152k pares)
n_points = len(d_rna_vec)
if n_points > 20000:
    idx = np.random.default_rng(SEED).choice(n_points, 20000, replace=False)
    x_plot, y_plot = d_rna_vec[idx], d_wsi_vec[idx]
else:
    x_plot, y_plot = d_rna_vec, d_wsi_vec

ax.scatter(x_plot, y_plot, s=4, alpha=0.15, color='steelblue', edgecolors='none')

# Línea de regresión (ajuste lineal simple)
z = np.polyfit(d_rna_vec, d_wsi_vec, 1)
x_line = np.linspace(d_rna_vec.min(), d_rna_vec.max(), 100)
ax.plot(x_line, np.polyval(z, x_line), color='firebrick', lw=2, label='Ajuste lineal')

ax.set_xlabel('Distancia RNA (1 − Pearson)')
ax.set_ylabel('Distancia WSI (coseno)')
ax.set_title(f"Test de Mantel: r = {mantel_result['r_observed']:.3f}, "
             f"p = {mantel_result['p_value']:.4f}")
ax.legend(loc='upper left')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'mantel_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

In [9]:
"""
============================================================
PASO 3: Entrenamiento MOFA+
============================================================
- Vistas: RNA (552 × n_genes), WSI (552 × WSI_FEAT_DIM)
- K inicial = 15 factores; ARD prior poda automáticamente
- Spike-and-slab prior sobre pesos (sparsity)
- Inferencia variacional con seed fijo
"""
print("="*70); print("PASO 3: Entrenamiento MOFA+"); print("="*70)

if not MOFA_MODEL_PATH.exists():
    ent = entry_point()

    ent.set_data_options(
        scale_views=False,    # ya centramos manualmente
        scale_groups=False
    )

    data_list = [
        [X_RNA_proc],   # vista 0: RNA
        [X_WSI_proc],   # vista 1: WSI
    ]
    view_names_mofa  = ['rna', 'wsi']
    group_names = ['all_patients']
    feature_names = [
        gene_names.tolist(),
        [f'{WSI_FEAT_PREFIX}_{i}' for i in range(WSI_FEAT_DIM)]
    ]
    sample_names = [patient_order]

    ent.set_data_matrix(
        data_list,
        views_names=view_names_mofa,
        groups_names=group_names,
        samples_names=sample_names,
        features_names=feature_names
    )

    ent.set_model_options(
        factors=15,
        spikeslab_weights=True,
        ard_weights=True,
        ard_factors=False,
    )

    ent.set_train_options(
        iter=1000,
        convergence_mode='medium',
        startELBO=1,
        freqELBO=10,
        dropR2=0.001,
        gpu_mode=False,
        verbose=False,
        seed=SEED
    )

    print("  Construyendo modelo MOFA...")
    ent.build()
    print("  Entrenando (esto toma varios minutos)...")
    ent.run()
    ent.save(outfile=str(MOFA_MODEL_PATH))
    print(f"  Modelo guardado: {MOFA_MODEL_PATH}")
else:
    print(f"  Modelo MOFA ya existe en {MOFA_MODEL_PATH}, se reutiliza")

PASO 3: Entrenamiento MOFA+
  Modelo MOFA ya existe en /home/dlaguna/Fusion/new/UNI_Parches/mofa_model.hdf5, se reutiliza


In [10]:
"""
============================================================
PASO 4: Extracción de outputs MOFA + descomposición de varianza
============================================================
- Factor scores Z (552 × K)
- Pesos W por vista
- R² por factor y vista (CORREGIDO: no multiplicar ×100 dos veces)
- Clasificación: factores compartidos vs específicos
"""
print("="*70); print("PASO 4: Extracción de outputs MOFA"); print("="*70)

with h5py.File(MOFA_MODEL_PATH, 'r') as f:
    Z     = np.array(f['expectations/Z/all_patients']).T       # (552, K)
    W_rna = np.array(f['expectations/W/rna']).T                # (n_genes, K)
    W_wsi = np.array(f['expectations/W/wsi']).T                # (WSI_FEAT_DIM, K)
    # CORREGIDO: el HDF5 guarda R² ya en porcentaje (0–100), NO fracción
    var_per_factor = np.array(f['variance_explained/r2_per_factor/all_patients']).T  # (K, 2)
    r2_total       = np.array(f['variance_explained/r2_total/all_patients'])         # (2,)

n_factors = Z.shape[1]
factor_names = [f'F{i+1}' for i in range(n_factors)]

print(f"  Z factor scores: {Z.shape}")
print(f"  W_rna weights:   {W_rna.shape}")
print(f"  W_wsi weights:   {W_wsi.shape}")
print(f"  Factores efectivos tras ARD: {n_factors}")

print(f"\n  R² total (varianza explicada):")
print(f"    RNA: {r2_total[0]:.2f}%")
print(f"    WSI: {r2_total[1]:.2f}%")

# Tabla de descomposición
var_df = pd.DataFrame(var_per_factor, columns=['R2_rna_pct', 'R2_wsi_pct'])
var_df['Factor'] = factor_names
var_df['shared']       = (var_df['R2_rna_pct'] > 1.0) & (var_df['R2_wsi_pct'] > 1.0)
var_df['rna_specific'] = (var_df['R2_rna_pct'] > 1.0) & (var_df['R2_wsi_pct'] < 0.5)
var_df['wsi_specific'] = (var_df['R2_wsi_pct'] > 1.0) & (var_df['R2_rna_pct'] < 0.5)
var_df = var_df[['Factor','R2_rna_pct','R2_wsi_pct','shared','rna_specific','wsi_specific']]

print(f"\n  Descomposición por factor:")
print(var_df.to_string(index=False, float_format='%.2f'))

n_shared   = var_df['shared'].sum()
n_rna_spec = var_df['rna_specific'].sum()
n_wsi_spec = var_df['wsi_specific'].sum()
print(f"\n  Compartidos (>1% ambas):              {n_shared}")
print(f"  RNA-específicos (>1% RNA, <0.5% WSI): {n_rna_spec}")
print(f"  WSI-específicos (>1% WSI, <0.5% RNA): {n_wsi_spec}")

# Guardar
var_df.to_csv(OUT_DIR / 'MOFA_variance_decomposition.csv', index=False)
np.save(OUT_DIR / 'MOFA_factor_scores.npy', Z)

PASO 4: Extracción de outputs MOFA
  Z factor scores: (552, 14)
  W_rna weights:   (604, 14)
  W_wsi weights:   (1024, 14)
  Factores efectivos tras ARD: 14

  R² total (varianza explicada):
    RNA: 43.29%
    WSI: 46.34%

  Descomposición por factor:
Factor  R2_rna_pct  R2_wsi_pct  shared  rna_specific  wsi_specific
    F1       32.90        2.29    True         False         False
    F2        0.02        7.82   False         False          True
    F3        0.02        6.75   False         False          True
    F4        5.20        0.04   False          True         False
    F5        0.04        4.53   False         False          True
    F6        0.06        4.02   False         False          True
    F7        1.06        2.78    True         False         False
    F8        0.95        2.75   False         False         False
    F9        2.19        1.50    True         False         False
   F10        0.01        3.49   False         False          True
   F11    

In [11]:
"""
============================================================
PASO 5: Asociación factores ↔ subtipo PAM50
============================================================
- ANOVA F-test por factor (scores vs subtipo)
- Corrección FDR Benjamini-Hochberg
- Verificación de ortogonalidad entre factores
"""
print("="*70); print("PASO 5: Asociación factores ↔ subtipo PAM50"); print("="*70)

# ---- ANOVA ----
assoc_rows = []
for k in range(n_factors):
    scores_by_sub = [Z[subtypes_str == s, k] for s in ['Basal', 'LumA', 'LumB']]
    F_stat, p_val = stats.f_oneway(*scores_by_sub)
    assoc_rows.append({
        'Factor': factor_names[k],
        'F_stat': F_stat,
        'p_value': p_val,
        'R2_rna_pct': var_per_factor[k, 0],
        'R2_wsi_pct': var_per_factor[k, 1],
    })
assoc_df = pd.DataFrame(assoc_rows)
reject, p_fdr, _, _ = multipletests(assoc_df['p_value'].values, alpha=0.05, method='fdr_bh')
assoc_df['p_FDR'] = p_fdr
assoc_df['significant_FDR05'] = reject

print(assoc_df.to_string(index=False, float_format='%.4g'))
print(f"\n  Factores asociados a subtipo (FDR<0.05): "
      f"{assoc_df['significant_FDR05'].sum()} de {n_factors}")

# ---- Ortogonalidad ----
factor_corr = np.corrcoef(Z.T)
max_offdiag = np.abs(factor_corr - np.eye(n_factors)).max()
print(f"\n  Ortogonalidad: máx |r| fuera de diagonal = {max_offdiag:.4f}")
print(f"  {'OK: todos <0.3' if max_offdiag < 0.3 else 'ATENCIÓN: hay pares >0.3 (revisar)'}")

# Guardar
assoc_df.to_csv(OUT_DIR / 'MOFA_factor_subtype_association.csv', index=False)

PASO 5: Asociación factores ↔ subtipo PAM50
Factor  F_stat    p_value  R2_rna_pct  R2_wsi_pct      p_FDR  significant_FDR05
    F1    3408 3.018e-310        32.9       2.295 4.225e-309               True
    F2   4.188    0.01566     0.01517       7.818     0.0274               True
    F3  0.7528     0.4715     0.01886       6.754     0.4715              False
    F4   98.41  2.979e-37       5.195      0.0359  2.085e-36               True
    F5   2.551    0.07894     0.03727       4.532     0.1005              False
    F6   1.426     0.2411     0.05793       4.025     0.2597              False
    F7   33.24  2.361e-14       1.058       2.776  8.264e-14               True
    F8   22.26  5.073e-10      0.9533       2.746   1.42e-09               True
    F9   38.14  3.103e-16       2.187       1.496  1.448e-15               True
   F10   2.648    0.07171     0.01337       3.495     0.1005              False
   F11   1.829     0.1616    0.003856       3.477     0.1886              Fa

In [12]:
"""
============================================================
PASO 6: GSEA sobre pesos RNA por factor (para ANEXO)
============================================================
- Prerank GSEA usando pesos W_rna como ranking de genes
- MSigDB Hallmark + KEGG + GO-BP
- Top 5 factores significativos
- Mínimo necesario para el anexo de la tesis
"""
print("="*70); print("PASO 6: GSEA (interpretación biológica — anexo)"); print("="*70)

sig_df = assoc_df[assoc_df['significant_FDR05']].sort_values('p_FDR')
gsea_results = {}

if HAS_GSEAPY and len(sig_df) > 0:
    gene_sets_list = ['MSigDB_Hallmark_2020', 'KEGG_2021_Human', 'GO_Biological_Process_2023']
    top_n_factors = min(5, len(sig_df))
    top_factors = sig_df.head(top_n_factors)['Factor'].tolist()

    for fname in top_factors:
        k = int(fname[1:]) - 1
        print(f"\n  GSEA para {fname} (R² RNA={var_per_factor[k,0]:.2f}%, "
              f"WSI={var_per_factor[k,1]:.2f}%):")

        gene_rank = pd.DataFrame({
            'gene': gene_names,
            'weight': W_rna[:, k]
        }).sort_values('weight', ascending=False)

        try:
            pre_res = gp.prerank(
                rnk=gene_rank,
                gene_sets=gene_sets_list,
                threads=4, min_size=5, max_size=500,
                permutation_num=1000, outdir=None,
                seed=SEED, verbose=False
            )
            top_paths = pre_res.res2d.sort_values('NES', key=lambda x: x.abs(),
                                                   ascending=False).head(10)
            print(f"    Top 10 vías enriquecidas (por |NES|):")
            for _, row in top_paths.iterrows():
                d = "+" if row['NES'] > 0 else "-"
                print(f"      {d} {row['Term'][:60]:60s} NES={row['NES']:+.2f}  "
                      f"FDR={row['FDR q-val']:.3f}")
            gsea_results[fname] = top_paths[['Term','NES','FDR q-val','Gene %']].to_dict(orient='records')
        except Exception as e:
            print(f"    ERROR en GSEA: {e}")
            gsea_results[fname] = None

    with open(OUT_DIR / 'MOFA_GSEA_results.json', 'w') as f:
        json.dump(gsea_results, f, indent=2, default=str)
    print(f"\n  Archivo: MOFA_GSEA_results.json")
else:
    print("  SKIP: gseapy no instalado o no hay factores significativos")

PASO 6: GSEA (interpretación biológica — anexo)

  GSEA para F1 (R² RNA=32.90%, WSI=2.29%):
    Top 10 vías enriquecidas (por |NES|):
      - GO_Biological_Process_2023__Intermediate Filament Organizati NES=-2.95  FDR=0.001
      - GO_Biological_Process_2023__Defense Response To Bacterium (G NES=-2.54  FDR=0.001
      + MSigDB_Hallmark_2020__Estrogen Response Early                NES=+2.46  FDR=0.001
      - GO_Biological_Process_2023__Granulocyte Chemotaxis (GO:00716 NES=-2.41  FDR=0.003
      - GO_Biological_Process_2023__Epithelium Development (GO:00604 NES=-2.41  FDR=0.002
      - GO_Biological_Process_2023__Neutrophil Chemotaxis (GO:003059 NES=-2.33  FDR=0.005
      - GO_Biological_Process_2023__Supramolecular Fiber Organizatio NES=-2.31  FDR=0.006
      - GO_Biological_Process_2023__Epidermis Development (GO:000854 NES=-2.29  FDR=0.006
      - GO_Biological_Process_2023__Response To Interleukin-1 (GO:00 NES=-2.17  FDR=0.017
      - GO_Biological_Process_2023__Chemokine-Mediated S

In [13]:
"""
============================================================
PASO 7: Clasificación supervisada sobre factor scores MOFA
============================================================
▸ Mismo diseño de búsqueda de hiperparámetros que Fases 1 y 2
▸ StratifiedKFold(5) para tunear HP (GridSearch/RandomizedSearch)
▸ RepeatedStratifiedKFold(5×3) para el reporte final (CV scores)
▸ Evaluación final en test set fijo (70/30 original)
▸ Clasificadores: KNN, RF, LR, SVM, XGBoost
▸ Scoring: F1-macro (prioriza equilibrio entre subtipos)
"""
print("="*70)
print("PASO 7: Clasificación supervisada (CV repetida + HP tuning)")
print("="*70)

# ---- Datos ----
Z_train, Z_test = Z[train_mask], Z[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"  Input: Z factor scores ({n_factors} factores)")
print(f"  Train: {Z_train.shape[0]}, Test: {Z_test.shape[0]}")

# ---- CV strategy (mismo diseño que Fases 1 y 2) ----
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)            # para tunear HP
cv_report = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)  # reporte final

# ---- Definición de clasificadores + grids (CORREGIDO) ----
# Formato: (search_type, estimator_or_pipeline, param_grid)
classifiers = {
    'KNN': (
        'grid',
        Pipeline([
            ('sc', StandardScaler()),
            ('clf', KNeighborsClassifier())
        ]),
        {
            'clf__n_neighbors': list(range(1, 31)),
            'clf__weights': ['uniform', 'distance'],
            'clf__metric': ['minkowski', 'cosine'],
            'clf__p': [1, 2, 3]
        }
    ),
    'Random Forest': (
        'random',
        RandomForestClassifier(random_state=SEED, n_jobs=-1),
        {
            'n_estimators': [100, 200, 300, 500],
            'max_depth': [None, 30, 40, 50],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 5],
            'max_features': [0.02, 0.05, 0.1, 0.15, 0.18],
            'criterion': ['gini', 'entropy']
        }
    ),
    'Logistic Regression': (
        'grid',
        Pipeline([
            ('sc', StandardScaler()),
            ('clf', LogisticRegression(max_iter=5000, random_state=SEED))
        ]),
        {
            'clf__C': [0.01, 0.1, 1, 10],
            'clf__penalty': ['l2'],
            'clf__solver': ['lbfgs']
        }
    ),
    'SVM': (
        'grid',
        Pipeline([
            ('sc', StandardScaler()),
            ('clf', SVC(random_state=SEED))
        ]),
        {
            'clf__C': [0.01, 0.1, 1, 10, 100],
            'clf__kernel': ['rbf', 'poly'],
            'clf__gamma': ['scale', 1, 0.1, 0.01, 0.001, 0.0001],
            'clf__degree': [2, 3, 4],
            'clf__coef0': [0.0, 0.1, 1.0]
        }
    ),
    'XGBoost': (
        'random',
        XGBClassifier(
            objective='multi:softprob', num_class=3,
            tree_method='hist', eval_metric='mlogloss',
            n_jobs=-1, random_state=SEED, verbosity=0
        ),
        {
            'n_estimators': stats.randint(200, 600),
            'max_depth': stats.randint(3, 7),
            'learning_rate': stats.uniform(0.03, 0.15),
            'subsample': stats.uniform(0.6, 0.3),
            'colsample_bytree': stats.uniform(0.4, 0.5),
            'min_child_weight': stats.randint(1, 6)
        }
    ),
 
}

# ---- Ejecutar búsqueda de HP + evaluación ----
mofa_results = []
mofa_preds   = {}
best_models  = {}

for name, (search_type, estimator, param_grid) in classifiers.items():
    print(f"\n  ── {name} ({search_type}SearchCV) ──")

    if search_type == 'grid':
        search = GridSearchCV(
            estimator, param_grid,
            cv=cv, scoring='f1_macro',
            n_jobs=-1, refit=True, verbose=0
        )
    else:  # 'random'
        search = RandomizedSearchCV(
            estimator, param_grid,
            n_iter=N_ITER_RANDOM,
            cv=cv, scoring='f1_macro',
            n_jobs=-1, refit=True, verbose=0,
            random_state=SEED
        )

    search.fit(Z_train, y_train)
    pred = search.predict(Z_test)

    f1m   = f1_score(y_test, pred, average='macro')
    f1p   = f1_score(y_test, pred, average=None)   # per-class
    acc   = accuracy_score(y_test, pred)
    cv_mean = search.best_score_
    cv_std  = search.cv_results_['std_test_score'][search.best_index_]

    mofa_results.append({
        'Modelo': name,
        'CV_F1_mean': cv_mean,
        'CV_F1_std': cv_std,
        'Test_F1_macro': f1m,
        'Test_Accuracy': acc,
        'F1_Basal': f1p[0],
        'F1_LumA': f1p[1],
        'F1_LumB': f1p[2],
    })
    mofa_preds[name]  = pred
    best_models[name] = search

    print(f"    Best CV F1-macro: {cv_mean:.3f} ± {cv_std:.3f}")
    print(f"    Test  F1-macro:   {f1m:.3f}  |  Acc: {acc:.3f}")
    print(f"    Per-class F1: Basal={f1p[0]:.3f}, LumA={f1p[1]:.3f}, LumB={f1p[2]:.3f}")
    # Top HP
    best_params = search.best_params_
    params_str = ', '.join([f'{k}={v}' for k, v in list(best_params.items())[:5]])
    print(f"    Best HP: {params_str}{'...' if len(best_params)>5 else ''}")

# ---- Reporte final: CV repetida (5×3) sobre todo el conjunto (n=552) ----
print("\n" + "-"*50)
print("Reporte CV repetida (5-fold × 3 repeats) — n=552 completo:")
print("-"*50)

scoring = {
    'F1_macro':        'f1_macro',
    'Balanced_acc':    'balanced_accuracy',
    'Precision_macro': 'precision_macro',
    'Recall_macro':    'recall_macro',
}

modelos_report = {name: search.best_estimator_ for name, search in best_models.items()}
filas_report, f1_folds = [], {}

for nombre, modelo in modelos_report.items():
    res = cross_validate(modelo, Z, y, cv=cv_report, scoring=scoring, n_jobs=-1)
    f1_folds[nombre] = res['test_F1_macro']
    fila = {'Modelo': nombre}
    for m in scoring:
        fila[m] = f"{res['test_'+m].mean():.3f} ± {res['test_'+m].std():.3f}"
    filas_report.append(fila)
    # Actualizar mofa_results con CV sobre 552
    for row in mofa_results:
        if row['Modelo'] == nombre:
            row['CV_F1_mean'] = res['test_F1_macro'].mean()
            row['CV_F1_std']  = res['test_F1_macro'].std()

tabla_report = pd.DataFrame(filas_report).set_index('Modelo').sort_values('F1_macro', ascending=False)
print(tabla_report.to_string())

# --- Test estadístico entre los dos mejores modelos ---
top2 = tabla_report.index[:2].tolist()
stat, p = wilcoxon(f1_folds[top2[0]], f1_folds[top2[1]])
print(f"\nWilcoxon {top2[0]} vs {top2[1]} (F1-macro por fold): p = {p:.3f}")
print("Diferencia significativa" if p < 0.05 else
      "Sin diferencia significativa -> rinden equivalente")

# ---- Tabla resumen ----
mofa_clf_df = pd.DataFrame(mofa_results).sort_values('Test_F1_macro', ascending=False)
print("\n" + "="*70)
print("RESUMEN CLASIFICACIÓN SOBRE MOFA FACTORS")
print("="*70)
print(mofa_clf_df.to_string(index=False, float_format='%.3f'))

best_name = mofa_clf_df.iloc[0]['Modelo']
best_f1   = mofa_clf_df.iloc[0]['Test_F1_macro']
print(f"\n  Mejor clasificador: {best_name} (Test F1-macro = {best_f1:.3f})")

# ---- Matrices de confusión ----
print("\n  Matrices de confusión (test set):")
for name in mofa_clf_df['Modelo'].values[:3]:  # top 3
    cm = confusion_matrix(y_test, mofa_preds[name])
    print(f"\n  {name}:")
    print(f"          {'   '.join(le.classes_)}")
    for i, row in enumerate(cm):
        print(f"    {le.classes_[i]:6s} {row}")

# ---- No supervisado: KMeans sobre factor scores ----
km = KMeans(n_clusters=3, n_init=50, random_state=SEED).fit(Z)
ari_mofa = adjusted_rand_score(y, km.labels_)
print(f"\n  ARI KMeans (k=3) sobre MOFA factors: {ari_mofa:.4f}")

# ---- Guardar ----
mofa_clf_df.to_csv(OUT_DIR / 'MOFA_classification_results.csv', index=False)

# Guardar best HP para cada modelo
hp_summary = {}
for name, search in best_models.items():
    hp_summary[name] = {
        'best_params': {k: str(v) for k, v in search.best_params_.items()},
        'best_cv_score': float(search.best_score_),
    }
with open(OUT_DIR / 'MOFA_best_hyperparameters.json', 'w') as f:
    json.dump(hp_summary, f, indent=2)

print("\n  Archivos: MOFA_classification_results.csv, MOFA_best_hyperparameters.json")

PASO 7: Clasificación supervisada (CV repetida + HP tuning)
  Input: Z factor scores (14 factores)
  Train: 386, Test: 166

  ── KNN (gridSearchCV) ──
    Best CV F1-macro: 0.847 ± 0.037
    Test  F1-macro:   0.784  |  Acc: 0.777
    Per-class F1: Basal=0.949, LumA=0.679, LumB=0.724
    Best HP: clf__metric=minkowski, clf__n_neighbors=28, clf__p=1, clf__weights=uniform

  ── Random Forest (randomSearchCV) ──
    Best CV F1-macro: 0.891 ± 0.024
    Test  F1-macro:   0.849  |  Acc: 0.843
    Per-class F1: Basal=0.990, LumA=0.764, LumB=0.793
    Best HP: n_estimators=100, min_samples_split=2, min_samples_leaf=1, max_features=0.18, max_depth=30...

  ── Logistic Regression (gridSearchCV) ──
    Best CV F1-macro: 0.873 ± 0.016
    Test  F1-macro:   0.907  |  Acc: 0.904
    Per-class F1: Basal=0.990, LumA=0.855, LumB=0.876
    Best HP: clf__C=1, clf__penalty=l2, clf__solver=lbfgs

  ── SVM (gridSearchCV) ──
    Best CV F1-macro: 0.888 ± 0.039
    Test  F1-macro:   0.895  |  Acc: 0.892
    Pe

In [14]:
"""
============================================================
PASO 8: Visualizaciones
============================================================
1. Heatmap de varianza explicada (R² por factor y vista)
2. Scatter top 2 factores + violin top 5
3. UMAP sobre factor scores
4. Ortogonalidad entre factores
5. Violin plots con significancia (top 6 factores)
6. Heatmap top genes por factor
"""
print("="*70); print("PASO 8: Visualizaciones"); print("="*70)

colors_sub = {'Basal': '#d62728', 'LumA': '#1f77b4', 'LumB': '#2ca02c'}

# ---- Fig 1: Variance heatmap ----
fig, ax = plt.subplots(figsize=(8, max(6, n_factors*0.35)))
sns.heatmap(
    var_per_factor, annot=True, fmt='.2f', cmap='YlOrRd',
    xticklabels=['RNA', 'WSI'],
    yticklabels=factor_names,
    ax=ax, cbar_kws={'label': 'R² (% varianza explicada)'},
    vmin=0, vmax=max(var_per_factor.max(), 5)
)
ax.set_title(f'MOFA+: descomposición de varianza\n'
             f'R² total RNA={r2_total[0]:.1f}%, WSI={r2_total[1]:.1f}% | '
             f'Extractor: {WSI_EXTRACTOR_NAME}', fontsize=11)
ax.set_ylabel('Factor')
plt.tight_layout()
plt.savefig(OUT_DIR / 'MOFA_variance_heatmap.png', dpi=140, bbox_inches='tight')
plt.close()
print("  ✓ MOFA_variance_heatmap.png")

# ---- Fig 2: Scatter top 2 + violin top 5 ----
sig_sorted = assoc_df[assoc_df['significant_FDR05']].sort_values('p_FDR')
if len(sig_sorted) >= 2:
    top_2 = sig_sorted.head(2)['Factor'].values
    k1 = int(top_2[0][1:]) - 1
    k2 = int(top_2[1][1:]) - 1

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    ax = axes[0]
    for s in ['Basal', 'LumA', 'LumB']:
        mask = subtypes_str == s
        ax.scatter(Z[mask, k1], Z[mask, k2], c=colors_sub[s], label=s, alpha=0.7, s=15)
    ax.set_xlabel(f'{top_2[0]} (R² RNA={var_per_factor[k1,0]:.1f}%, WSI={var_per_factor[k1,1]:.1f}%)')
    ax.set_ylabel(f'{top_2[1]} (R² RNA={var_per_factor[k2,0]:.1f}%, WSI={var_per_factor[k2,1]:.1f}%)')
    ax.set_title('Top 2 factores asociados a subtipo PAM50'); ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1]
    top_5_f = sig_sorted.head(5)['Factor'].values
    violin_data = []
    for fname in top_5_f:
        k = int(fname[1:]) - 1
        for s in ['Basal', 'LumA', 'LumB']:
            for v in Z[subtypes_str == s, k]:
                violin_data.append({'Factor': fname, 'Subtipo': s, 'Score': v})
    vio_df = pd.DataFrame(violin_data)
    sns.violinplot(data=vio_df, x='Factor', y='Score', hue='Subtipo',
                   palette=colors_sub, ax=ax, inner='quartile')
    ax.set_title('Scores top 5 factores'); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'MOFA_factor_scatter.png', dpi=140, bbox_inches='tight')
    plt.close()
    print("  ✓ MOFA_factor_scatter.png")

# ---- Fig 3: UMAP ----
if HAS_UMAP:
    fig, ax = plt.subplots(figsize=(8, 7))
    reducer = umap_lib.UMAP(n_neighbors=30, min_dist=0.1, random_state=SEED)
    coords = reducer.fit_transform(Z)
    for s in ['Basal', 'LumA', 'LumB']:
        mask = subtypes_str == s
        ax.scatter(coords[mask, 0], coords[mask, 1], c=colors_sub[s], label=s, alpha=0.7, s=15)
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
    ax.set_title(f'UMAP de {n_factors} factores MOFA — ARI(KMeans)={ari_mofa:.3f}', fontsize=11)
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'MOFA_UMAP.png', dpi=140, bbox_inches='tight')
    plt.close()
    print("  ✓ MOFA_UMAP.png")

# ---- Fig 4: Ortogonalidad ----
fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(factor_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            xticklabels=factor_names, yticklabels=factor_names, ax=ax,
            vmin=-1, vmax=1, cbar_kws={'label':'Pearson r'})
ax.set_title('Ortogonalidad entre factores MOFA', fontsize=11)
plt.tight_layout()
plt.savefig(OUT_DIR / 'MOFA_factor_orthogonality.png', dpi=140, bbox_inches='tight')
plt.close()
print("  ✓ MOFA_factor_orthogonality.png")

# ---- Fig 5: Violin con significancia (top 6) ----
top_6 = sig_sorted.head(6)['Factor'].tolist()
n_plots = len(top_6)
if n_plots > 0:
    ncols = min(3, n_plots); nrows = (n_plots + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.5*nrows))
    axes = np.array(axes).flatten()
    for i, fname in enumerate(top_6):
        ax = axes[i]
        k = int(fname[1:]) - 1
        data_plot = []
        for s in ['Basal', 'LumA', 'LumB']:
            for v in Z[subtypes_str == s, k]:
                data_plot.append({'Subtipo': s, 'Score': v})
        df_p = pd.DataFrame(data_plot)
        sns.violinplot(data=df_p, x='Subtipo', y='Score', ax=ax,
                       palette=colors_sub, inner=None, cut=0)
        sns.stripplot(data=df_p, x='Subtipo', y='Score', ax=ax,
                      color='black', size=2, alpha=0.4, jitter=True)
        # Pairwise t-tests con Bonferroni
        groups = [Z[subtypes_str == s, k] for s in ['Basal', 'LumA', 'LumB']]
        F, p = stats.f_oneway(*groups)
        pairs_lbl = [(0,1,'B-LA'), (0,2,'B-LB'), (1,2,'LA-LB')]
        ann = []
        for ii, jj, lbl in pairs_lbl:
            _, p_pair = stats.ttest_ind(groups[ii], groups[jj])
            p_adj = min(p_pair * 3, 1)
            sig = '***' if p_adj<0.001 else '**' if p_adj<0.01 else '*' if p_adj<0.05 else 'ns'
            ann.append(f'{lbl}:{sig}')
        ax.set_title(f'{fname} (R² RNA={var_per_factor[k,0]:.1f}%, WSI={var_per_factor[k,1]:.1f}%)\n'
                     f'p={p:.1e} | {" ".join(ann)}', fontsize=9)
        ax.set_ylabel('Factor score'); ax.grid(axis='y', alpha=0.3)
    for j in range(n_plots, len(axes)): axes[j].axis('off')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'MOFA_violin_factors.png', dpi=140, bbox_inches='tight')
    plt.close()
    print("  ✓ MOFA_violin_factors.png")

# ---- Fig 6: Heatmap top genes por factor ----
top_3_f = sig_sorted.head(3)['Factor'].tolist()
if len(top_3_f) > 0:
    X_RNA_centered = X_RNA_raw - X_RNA_raw.mean(axis=0, keepdims=True)
    sub_order = ['Basal', 'LumA', 'LumB']
    order_idx = np.concatenate([np.where(subtypes_str == s)[0] for s in sub_order])

    fig, axes_h = plt.subplots(1, len(top_3_f), figsize=(7*len(top_3_f), 8))
    if len(top_3_f) == 1: axes_h = [axes_h]
    for ax, fname in zip(axes_h, top_3_f):
        k = int(fname[1:]) - 1
        abs_w = np.abs(W_rna[:, k])
        top_idx = np.argsort(-abs_w)[:30]
        sign_order = np.argsort(-W_rna[top_idx, k])
        top_idx_sorted = top_idx[sign_order]
        sub_X = X_RNA_centered[np.ix_(order_idx, top_idx_sorted)].T
        sub_names = gene_names[top_idx_sorted]
        sub_X_z = (sub_X - sub_X.mean(axis=1, keepdims=True)) / (sub_X.std(axis=1, keepdims=True) + 1e-8)
        sub_X_z = np.clip(sub_X_z, -3, 3)
        sns.heatmap(sub_X_z, cmap='RdBu_r', center=0, ax=ax,
                    yticklabels=sub_names, xticklabels=False,
                    cbar_kws={'label':'z-score'}, vmin=-3, vmax=3)
        boundaries = np.cumsum([np.sum(subtypes_str == s) for s in sub_order[:-1]])
        for b in boundaries: ax.axvline(b, color='black', lw=1.5)
        centers, prev = [], 0
        for s in sub_order:
            n = np.sum(subtypes_str == s); centers.append(prev + n/2); prev += n
        ax.set_xticks(centers); ax.set_xticklabels(sub_order, rotation=0)
        ax.set_title(f'{fname}: top 30 genes\nR² RNA={var_per_factor[k,0]:.1f}%, '
                     f'WSI={var_per_factor[k,1]:.1f}%', fontsize=10)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'MOFA_top_genes_heatmap.png', dpi=140, bbox_inches='tight')
    plt.close()
    print("  ✓ MOFA_top_genes_heatmap.png")

# ---- Fig 7: GSEA bar chart ----
if HAS_GSEAPY and gsea_results:
    plot_factors = [f for f in list(gsea_results.keys())[:3] if gsea_results[f]]
    if plot_factors:
        fig, axes_g = plt.subplots(1, len(plot_factors), figsize=(6*len(plot_factors), 6), sharey=False)
        if len(plot_factors) == 1: axes_g = [axes_g]
        for ax, fname in zip(axes_g, plot_factors):
            df_paths = pd.DataFrame(gsea_results[fname]).head(10)
            df_paths['neg_log10_fdr'] = -np.log10(
                df_paths['FDR q-val'].astype(float).replace(0, 1e-4))
            df_paths = df_paths.sort_values('NES', key=lambda x: x.abs())
            colors = ['#d62728' if x > 0 else '#1f77b4' for x in df_paths['NES']]
            ax.barh(range(len(df_paths)), df_paths['NES'], color=colors, edgecolor='black')
            ax.set_yticks(range(len(df_paths)))
            term_labels = [t[:50]+'...' if len(t) > 50 else t for t in df_paths['Term']]
            ax.set_yticklabels(term_labels, fontsize=8)
            ax.axvline(0, color='black', lw=0.5)
            ax.set_xlabel('NES')
            ax.set_title(f'{fname}: top vías', fontsize=10)
            ax.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.savefig(OUT_DIR / 'MOFA_GSEA_top_factors.png', dpi=140, bbox_inches='tight')
        plt.close()
        print("  ✓ MOFA_GSEA_top_factors.png")

print("\n  Todas las figuras generadas.")

PASO 8: Visualizaciones
  ✓ MOFA_variance_heatmap.png
  ✓ MOFA_factor_scatter.png
  ✓ MOFA_UMAP.png
  ✓ MOFA_factor_orthogonality.png
  ✓ MOFA_violin_factors.png
  ✓ MOFA_top_genes_heatmap.png
  ✓ MOFA_GSEA_top_factors.png

  Todas las figuras generadas.


In [15]:
rna = pd.read_csv('/home/dlaguna/filtered_normalized_data_INC_23.csv', sep=';')
wsi = pd.read_csv('/home/dlaguna/TCGA_work/UNI/Validacion_UNI_Parches/meanpool_features/UNI_patches_meanpool_INC23.csv')

for r, w in zip(rna.iloc[0, 1:].values, wsi['ID'].tolist()):
    print(r, "|", w)

X116BBp-C_S11 | 02- M4753-13
X260B-Bp-C_S5 | 03- M2588-19 A1
X065BBn-C_S9 | 08- M1225-17
X175B-Bp-C_S11 | 11- 16958-17
X155B-Bn-C_S1 | 12- 3092-17  2
X145B-Bp-C_S1 | 13- 14641-16  
X125B-Bn_S10 | 14- M0675-17
X084B-Bn_S10 | 16- 12134-16  2
X188B-Bn-C_S16 | 19- M5820-16  1
F70B-Bn_S15 | 21- 8189-14
X117B-Bp_S3 | 22- M4047- 14
F20B-Bn-C_S13 | 24- 16591-16
X051B-Bn-C_S4 | 26- M1699-17
F31B-Bn_S10 | 27-M1273-15  A
X314B-Bn_S9 | 28- M4757-16
X239B-Bn_S1 | 29- M4106-16  2
X018B-Bp-C_S12 | 30- M2951-14  2
X219B-Bn_S16 | 32- 5161-18
X142B-Bp-C_S2 | 34- M1925-18
X261B-Bp-C_S15 | 36- M3037-19
X115-B_S7 | 37- 10267-19
X295-B_S8 | 40- M4954-16  3
X066-B_S3 | 42- M5026-19


In [16]:
coin = pd.read_csv('/home/dlaguna/coincidencias_pacientes_inc.txt', sep=None, engine='python')
print(coin.shape)
print(coin.head())

(23, 2)
       SampleName  slide_ID
0   F20B.Bn.C_S13  M4753-13
1     F31B.Bn_S10  M2588-14
2     F70B.Bn_S15  M1225-17
3  X018B.Bp.C_S12  16958-17
4   X051B.Bn.C_S4   3092-17


In [17]:
"""
============================================================
PASO 9: Validación externa — INC Bogotá (Luminal B)
============================================================
- Proyectar datos INC al espacio de factores MOFA
- Evaluar clasificador ganador sobre pacientes INC
- Evaluación de transferibilidad hacia Luminal B
  (NO validación multiclase completa)
"""
# ── PASO 9: Validación externa — INC Bogotá ──────────────────────────────────

# ---- Cargar RNA INC ----
inc_rna_raw      = pd.read_csv(INC_RNA_PATH, sep=';')
inc_rna_patients = inc_rna_raw.iloc[0, 1:].values
inc_rna_genes    = inc_rna_raw.iloc[2:, 0].values
inc_rna_vals     = inc_rna_raw.iloc[2:, 1:].astype(float).values
inc_rna_df       = pd.DataFrame(inc_rna_vals.T,
                                index=inc_rna_patients,
                                columns=inc_rna_genes)
print(f"  RNA INC: {inc_rna_df.shape}")

# ---- Cargar WSI INC (UNI features) ----
import re
COIN_PATH = '/home/dlaguna/coincidencias_pacientes_inc.txt'   # ← ajusta la ruta si es otra

inc_wsi_df    = pd.read_csv(INC_WSI_PATH)
feat_cols_wsi = [str(i) for i in range(WSI_FEAT_DIM)]

# ---- Emparejamiento RNA ↔ WSI por ID (no por posición) ----
def key_paciente(s):
    """X018B.Bp.C_S12 y X018B-Bp-C_S12 -> misma clave."""
    return re.sub(r'[.\-]', '-', str(s).strip())

def key_lamina(s, es_wsi=False):
    """'22- M4047- 14' / 'M4047-14' / '14- M0675-17' / 'M675-17' -> misma clave."""
    s = str(s).strip()
    if es_wsi:
        s = re.sub(r'^\d+-\s*', '', s)        # quita prefijo "22- " (solo en WSI)
    s = re.sub(r'-\s+', '-', s)               # "M4047- 14" -> "M4047-14"
    s = s.split()[0]                          # quita sufijos " A1", " 2"
    return re.sub(r'^(M?)0+', r'\1', s.upper())   # "M0675" -> "M675", "08189" -> "8189"

coin = pd.read_csv(COIN_PATH, sep=None, engine='python')
pac_a_lamina = {key_paciente(p): key_lamina(l) for p, l in zip(coin['SampleName'], coin['slide_ID'])}
# Correcciones manuales de erratas en el archivo de coincidencias (confirmadas con el INC)
CORRECCIONES = {'X065BBn-C_S9': '14641-16',
                'F31B-Bn_S10':  'M2588-19'}
pac_a_lamina.update({key_paciente(p): key_lamina(l) for p, l in CORRECCIONES.items()})

inc_wsi_df['key'] = inc_wsi_df['ID'].map(lambda s: key_lamina(s, es_wsi=True))
assert inc_wsi_df['key'].is_unique, "Hay láminas WSI duplicadas tras normalizar"

pacientes = [key_paciente(p) for p in inc_rna_patients]
sin_coin  = [p for p in pacientes if p not in pac_a_lamina]
assert not sin_coin, f"Pacientes RNA sin fila en el archivo de coincidencias: {sin_coin}"

orden_laminas = [pac_a_lamina[p] for p in pacientes]
sin_wsi = [(p, l) for p, l in zip(pacientes, orden_laminas) if l not in set(inc_wsi_df['key'])]
assert not sin_wsi, f"Láminas del archivo de coincidencias que no están en WSI: {sin_wsi}"

inc_wsi_df    = inc_wsi_df.set_index('key').loc[orden_laminas].reset_index()
inc_wsi_ids   = inc_wsi_df['ID'].tolist()
X_inc_wsi_raw = inc_wsi_df[feat_cols_wsi].astype(float).values
n_inc         = len(inc_rna_patients)

print(f"  WSI INC: {X_inc_wsi_raw.shape}")
print("  Emparejamiento RNA | WSI:")
for p, w in zip(inc_rna_patients, inc_wsi_ids):
    print(f"    {p:18s} | {w}")

# ---- Preprocesar RNA: alinear y centrar con media de TCGA ----
common_genes  = [g for g in gene_names if g in inc_rna_df.columns]
missing_genes = [g for g in gene_names if g not in inc_rna_df.columns]
print(f"  Genes comunes: {len(common_genes)} | Faltantes: {len(missing_genes)}")

tcga_gene_means = X_RNA_raw.mean(axis=0)
gene_to_idx     = {g: i for i, g in enumerate(gene_names)}

X_inc_rna = np.zeros((n_inc, len(gene_names)))  # genes ausentes → 0 tras centrado
for g in common_genes:
    idx = gene_to_idx[g]
    # Acceso posicional en lugar de por ID para evitar KeyError
    X_inc_rna[:, idx] = inc_rna_df[g].values - tcga_gene_means[idx]
print(f"  RNA INC procesada: {X_inc_rna.shape}")

# ---- Preprocesar WSI: L2-normalizar y centrar con media de TCGA ----
X_inc_wsi_l2   = normalize(X_inc_wsi_raw, norm='l2', axis=1)
tcga_wsi_means = X_WSI_l2.mean(axis=0)
X_inc_wsi      = X_inc_wsi_l2 - tcga_wsi_means
print(f"  WSI INC procesada: {X_inc_wsi.shape}")

# ---- Proyección al espacio de factores MOFA ----
W_comb     = W_rna.T @ W_rna + W_wsi.T @ W_wsi
W_comb_inv = np.linalg.pinv(W_comb)
Z_inc      = (X_inc_rna @ W_rna + X_inc_wsi @ W_wsi) @ W_comb_inv
print(f"  Z_inc proyectado: {Z_inc.shape}")  # debe ser (23, 14)

# ---- Predecir con todos los modelos ----
lumb_idx = le.transform(['LumB'])[0]
filas = []
for nombre, search in best_models.items():
    pred        = search.predict(Z_inc)
    pred_labels = le.inverse_transform(pred)
    n_correct   = (pred == lumb_idx).sum()
    filas.append({
        "Modelo":            nombre,
        "LumB correctas":    int(n_correct),
        "Total":             n_inc,
        "Sensibilidad LumB": round(n_correct / n_inc, 3),
        "Distribución":      dict(pd.Series(pred_labels).value_counts())
    })

tabla_inc = pd.DataFrame(filas).set_index("Modelo")
print("\n===== Sensibilidad LumB — INC (n=23) — MOFA+ =====")
print(tabla_inc.to_string())

  RNA INC: (23, 601)
  WSI INC: (23, 1024)
  Emparejamiento RNA | WSI:
    X116BBp-C_S11      | 21- 8189-14
    X260B-Bp-C_S5      | 36- M3037-19
    X065BBn-C_S9       | 13- 14641-16  
    X175B-Bp-C_S11     | 29- M4106-16  2
    X155B-Bn-C_S1      | 28- M4757-16
    X145B-Bp-C_S1      | 27-M1273-15  A
    X125B-Bn_S10       | 24- 16591-16
    X084B-Bn_S10       | 16- 12134-16  2
    X188B-Bn-C_S16     | 30- M2951-14  2
    F70B-Bn_S15        | 08- M1225-17
    X117B-Bp_S3        | 22- M4047- 14
    F20B-Bn-C_S13      | 02- M4753-13
    X051B-Bn-C_S4      | 12- 3092-17  2
    F31B-Bn_S10        | 03- M2588-19 A1
    X314B-Bn_S9        | 42- M5026-19
    X239B-Bn_S1        | 34- M1925-18
    X018B-Bp-C_S12     | 11- 16958-17
    X219B-Bn_S16       | 32- 5161-18
    X142B-Bp-C_S2      | 26- M1699-17
    X261B-Bp-C_S15     | 37- 10267-19
    X115-B_S7          | 19- M5820-16  1
    X295-B_S8          | 40- M4954-16  3
    X066-B_S3          | 14- M0675-17
  Genes comunes: 595 | Faltantes

In [18]:
missing = [g for g in gene_names if g not in inc_rna_df.columns]
print(f"Genes faltantes ({len(missing)}):")
for g in missing:
    idx = gene_to_idx[g]
    print(f"  {g:15s}  peso MOFA F1: {W_rna[idx,0]:.4f}")

Genes faltantes (9):
  SPOCK2           peso MOFA F1: -0.3700
  NEURL1           peso MOFA F1: 0.4863
  ADAMDEC1         peso MOFA F1: -0.3107
  FGFBP1           peso MOFA F1: -0.6579
  LY6D             peso MOFA F1: -0.9130
  TAP1             peso MOFA F1: -0.3325
  CST5             peso MOFA F1: 0.6286
  C6orf15          peso MOFA F1: -0.7313
  OR2I1P           peso MOFA F1: -0.4116


In [19]:
"""
============================================================
PASO 3b: Modelos MOFA+ de vista única (RNA-solo, WSI-solo)
============================================================
- Análisis complementario de interpretabilidad (anexo)
- Caracteriza la contribución individual de cada modalidad
"""
print("="*70); print("PASO 3b: MOFA-RNA-solo y MOFA-WSI-solo"); print("="*70)

MOFA_RNA_PATH = OUT_DIR / 'mofa_model_rna_solo.hdf5'
MOFA_WSI_PATH = OUT_DIR / 'mofa_model_wsi_solo.hdf5'

# ── MOFA-RNA-solo ──────────────────────────────────────────────────────────
if not MOFA_RNA_PATH.exists():
    ent_rna = entry_point()

    ent_rna.set_data_options(
        scale_views=False,
        scale_groups=False
    )

    ent_rna.set_data_matrix(
        [[X_RNA_proc]],                       # única vista: RNA
        views_names=['rna'],
        groups_names=['all_patients'],
        samples_names=[patient_order],
        features_names=[gene_names.tolist()]
    )

    ent_rna.set_model_options(
        factors=15,
        spikeslab_weights=True,
        ard_weights=True,
        ard_factors=False,
    )

    ent_rna.set_train_options(
        iter=1000,
        convergence_mode='medium',
        startELBO=1,
        freqELBO=10,
        dropR2=0.001,
        gpu_mode=False,
        verbose=False,
        seed=SEED
    )

    print("  Entrenando MOFA-RNA-solo...")
    ent_rna.build()
    ent_rna.run()
    ent_rna.save(outfile=str(MOFA_RNA_PATH))
    print(f"  Modelo guardado: {MOFA_RNA_PATH}")
else:
    print(f"  MOFA-RNA-solo ya existe en {MOFA_RNA_PATH}, se reutiliza")

# ── MOFA-WSI-solo ──────────────────────────────────────────────────────────
# Requiere parámetros relajados: la config estándar colapsa el posterior
# (ARD elimina todos los factores). Se desactiva dropR2 y se usa
# convergence_mode='fast' para permitir convergencia con varianza baja.
if not MOFA_WSI_PATH.exists():
    ent_wsi = entry_point()

    ent_wsi.set_data_options(
        scale_views=False,
        scale_groups=False
    )

    ent_wsi.set_data_matrix(
        [[X_WSI_proc]],                       # única vista: WSI
        views_names=['wsi'],
        groups_names=['all_patients'],
        samples_names=[patient_order],
        features_names=[[f'{WSI_FEAT_PREFIX}_{i}' for i in range(WSI_FEAT_DIM)]]
    )

    ent_wsi.set_model_options(
        factors=10,              # menos factores iniciales
        spikeslab_weights=True,
        ard_weights=True,
        ard_factors=False,
    )

    ent_wsi.set_train_options(
        iter=1000,
        convergence_mode='fast',  # relajado, evita colapso
        startELBO=1,
        freqELBO=10,
        dropR2=None,               # desactiva poda automática por R²
        gpu_mode=False,
        verbose=False,
        seed=SEED
    )

    print("\n  Entrenando MOFA-WSI-solo (parámetros relajados)...")
    ent_wsi.build()
    ent_wsi.run()
    ent_wsi.save(outfile=str(MOFA_WSI_PATH))
    print(f"  Modelo guardado: {MOFA_WSI_PATH}")
else:
    print(f"  MOFA-WSI-solo ya existe en {MOFA_WSI_PATH}, se reutiliza")


# ── Extracción de outputs para ambos modelos ────────────────────────────────
def extraer_outputs_mofa(path, view_key):
    with h5py.File(path, 'r') as f:
        Z_solo = np.array(f['expectations/Z/all_patients']).T
        W_solo = np.array(f[f'expectations/W/{view_key}']).T
        r2_total_solo = np.array(f['variance_explained/r2_total/all_patients'])
    return Z_solo, W_solo, r2_total_solo

Z_rna_solo, W_rna_solo, r2_rna_solo = extraer_outputs_mofa(MOFA_RNA_PATH, 'rna')
Z_wsi_solo, W_wsi_solo, r2_wsi_solo = extraer_outputs_mofa(MOFA_WSI_PATH, 'wsi')

print(f"\n  MOFA-RNA-solo: {Z_rna_solo.shape[1]} factores | R² = {r2_rna_solo[0]:.2f}%")
print(f"  MOFA-WSI-solo: {Z_wsi_solo.shape[1]} factores | R² = {r2_wsi_solo[0]:.2f}%")

PASO 3b: MOFA-RNA-solo y MOFA-WSI-solo
  MOFA-RNA-solo ya existe en /home/dlaguna/Fusion/new/UNI_Parches/mofa_model_rna_solo.hdf5, se reutiliza
  MOFA-WSI-solo ya existe en /home/dlaguna/Fusion/new/UNI_Parches/mofa_model_wsi_solo.hdf5, se reutiliza

  MOFA-RNA-solo: 14 factores | R² = 58.44%
  MOFA-WSI-solo: 10 factores | R² = 0.00%


In [23]:
from scipy.stats import f_oneway
from statsmodels.stats.multitest import multipletests

def tabla_factores(Z, y, nombre):
    filas = []
    for k in range(Z.shape[1]):
        grupos = [Z[np.asarray(y) == c, k] for c in np.unique(y)]
        F, p = f_oneway(*grupos)
        filas.append({"Factor": f"F{k+1}", "F_stat": round(F, 3), "p_value": p})
    df = pd.DataFrame(filas)
    df["p_FDR"] = multipletests(df["p_value"], method="fdr_bh")[1]
    df["sig_FDR05"] = df["p_FDR"] < 0.05
    print(f"\n{nombre} ({Z.shape[1]} factores)")
    print(df.to_string(index=False))
    return df

tab_rna = tabla_factores(Z_rna_solo, y, "MOFA-RNA-solo")
tab_wsi = tabla_factores(Z_wsi_solo, y, "MOFA-WSI-solo")


MOFA-RNA-solo (14 factores)
Factor   F_stat       p_value         p_FDR  sig_FDR05
    F1 3528.764 4.235594e-314 5.929832e-313       True
    F2  171.247  1.603496e-58  1.122448e-57       True
    F3    1.491  2.259784e-01  3.954622e-01      False
    F4    0.045  9.560016e-01  9.652901e-01      False
    F5   12.937  3.235952e-06  1.132583e-05       True
    F6    7.308  7.376974e-04  2.065553e-03       True
    F7    0.035  9.652901e-01  9.652901e-01      False
    F8    0.928  3.959502e-01  5.543302e-01      False
    F9    4.369  1.310576e-02  3.058010e-02       True
   F10    0.533  5.871500e-01  7.472818e-01      False
   F11   16.145  1.537041e-07  7.172857e-07       True
   F12    3.807  2.281408e-02  4.562817e-02       True
   F13    0.037  9.636489e-01  9.652901e-01      False
   F14    1.193  3.039704e-01  4.728428e-01      False

MOFA-WSI-solo (10 factores)
Factor  F_stat  p_value    p_FDR  sig_FDR05
    F1  13.741 0.000002 0.000002       True
    F2  13.746 0.000001 0.000

In [24]:
"""
============================================================
PASO 3c: Clasificación supervisada sobre MOFA-RNA-solo y MOFA-WSI-solo
============================================================
- Mismo protocolo que el modelo multimodal: 5 clasificadores,
  GridSearch/RandomizedSearch + CV repetida
- Resultados para anexo (comparación unimodal vs multimodal)
"""
print("="*70); print("PASO 3c: Clasificación sobre espacios unimodales"); print("="*70)

# Reutiliza el diccionario 'classifiers' ya definido (sin MLP)
# y train_mask/test_mask ya definidos en PASO 1

def clasificar_espacio_latente(Z_espacio, nombre_espacio):
    Z_train = Z_espacio[train_mask]
    Z_test  = Z_espacio[test_mask]

    resultados = []
    preds_dict = {}
    modelos_ajustados = {}

    for name, (search_type, estimator, param_grid) in classifiers.items():
        if search_type == 'grid':
            search = GridSearchCV(estimator, param_grid, cv=cv,
                                   scoring='f1_macro', n_jobs=-1, refit=True)
        else:
            search = RandomizedSearchCV(estimator, param_grid, n_iter=N_ITER_RANDOM,
                                         cv=cv, scoring='f1_macro', n_jobs=-1,
                                         refit=True, random_state=SEED)

        search.fit(Z_train, y_train)
        pred = search.predict(Z_test)

        f1m = f1_score(y_test, pred, average='macro')
        f1p = f1_score(y_test, pred, average=None)
        acc = accuracy_score(y_test, pred)

        resultados.append({
            'Modelo': name,
            'Espacio': nombre_espacio,
            'Test_F1_macro': f1m,
            'Test_Accuracy': acc,
            'F1_Basal': f1p[0], 'F1_LumA': f1p[1], 'F1_LumB': f1p[2],
        })
        preds_dict[name] = pred
        modelos_ajustados[name] = search

    df = pd.DataFrame(resultados).sort_values('Test_F1_macro', ascending=False)
    return df, preds_dict, modelos_ajustados

# ---- Clasificar sobre cada espacio ----
print("\n  Clasificando sobre MOFA-RNA-solo...")
df_rna_solo, preds_rna_solo, modelos_rna_solo = clasificar_espacio_latente(
    Z_rna_solo, 'RNA-solo'
)
print(df_rna_solo.to_string(index=False, float_format='%.3f'))

print("\n  Clasificando sobre MOFA-WSI-solo...")
df_wsi_solo, preds_wsi_solo, modelos_wsi_solo = clasificar_espacio_latente(
    Z_wsi_solo, 'WSI-solo'
)
print(df_wsi_solo.to_string(index=False, float_format='%.3f'))

# ---- Tabla comparativa: unimodal RNA vs unimodal WSI vs multimodal ----
best_rna_solo = df_rna_solo.iloc[0]
best_wsi_solo = df_wsi_solo.iloc[0]
best_multimodal = mofa_clf_df.iloc[0]   # ya calculado en PASO 7

comparacion_unimodal_mofa = pd.DataFrame([
    {'Espacio': 'MOFA-RNA-solo', 'Modelo': best_rna_solo['Modelo'],
     'F1_macro': best_rna_solo['Test_F1_macro']},
    {'Espacio': 'MOFA-WSI-solo', 'Modelo': best_wsi_solo['Modelo'],
     'F1_macro': best_wsi_solo['Test_F1_macro']},
    {'Espacio': 'MOFA-multimodal', 'Modelo': best_multimodal['Modelo'],
     'F1_macro': best_multimodal['Test_F1_macro']},
])
print("\n===== Comparación: MOFA unimodal vs multimodal =====")
print(comparacion_unimodal_mofa.to_string(index=False, float_format='%.3f'))

# ---- Guardar ----
df_rna_solo.to_csv(OUT_DIR / 'MOFA_RNA_solo_classification.csv', index=False)
df_wsi_solo.to_csv(OUT_DIR / 'MOFA_WSI_solo_classification.csv', index=False)
comparacion_unimodal_mofa.to_csv(OUT_DIR / 'MOFA_comparacion_unimodal_multimodal.csv', index=False)

PASO 3c: Clasificación sobre espacios unimodales

  Clasificando sobre MOFA-RNA-solo...
             Modelo  Espacio  Test_F1_macro  Test_Accuracy  F1_Basal  F1_LumA  F1_LumB
      Random Forest RNA-solo          0.896          0.892     0.990    0.839    0.857
Logistic Regression RNA-solo          0.895          0.892     0.980    0.847    0.860
            XGBoost RNA-solo          0.890          0.886     0.990    0.835    0.845
                SVM RNA-solo          0.884          0.880     0.980    0.832    0.840
                KNN RNA-solo          0.832          0.825     0.970    0.761    0.767

  Clasificando sobre MOFA-WSI-solo...
             Modelo  Espacio  Test_F1_macro  Test_Accuracy  F1_Basal  F1_LumA  F1_LumB
Logistic Regression WSI-solo          0.446          0.452     0.482    0.504    0.351
                SVM WSI-solo          0.417          0.446     0.438    0.538    0.275
            XGBoost WSI-solo          0.411          0.416     0.358    0.472    0.404
   